# YOLO26 车辆检测 - v3 摩托车微调
基于 v2 权重 + 新增 motorcycle 类别
冻结大部分层，只训练最后几层 → 快

In [1]:
from ultralytics import YOLO
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

model = YOLO('E:/dev/yjwlYOLO/dev/runs/train_v2/weights/5060ti1.pt')
print(f'原模型类别: {model.names}')

原模型类别: {0: 'car', 1: 'bus', 2: 'van', 3: 'truck', 4: 'others'}


In [2]:
# 微调: 冻结前面的层，只训练最后的检测头
# v2 已经会识别 car/bus/van/truck/others
# 只需要让检测头学会 motorcycle
results = model.train(
    data='E:/dev/yjwlYOLO/dev/dataset/data.yaml',
    epochs=15,          # 微调不需要太多轮
    imgsz=640,
    batch=32,
    device=0,
    workers=2,
    freeze=20,          # 冻结前20层，只训练检测头
    patience=5,         # 5轮没提升就早停
    lr0=0.001,          # 小学习率
    project='E:/dev/yjwlYOLO/dev/runs',
    name='train_v3_moto',
    exist_ok=True,
)

New https://pypi.org/project/ultralytics/8.4.66 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.48  Python-3.13.5 torch-2.8.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:/dev/yjwlYOLO/dev/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=20, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=E:/dev/yjwlYOLO/dev/runs/train_v2/weigh

In [3]:
model = YOLO('E:/dev/yjwlYOLO/dev/runs/train_v3_moto/weights/best.pt')
results = model.val(data='E:/dev/yjwlYOLO/dev/dataset/data.yaml', device='cpu')

print(f'mAP50:     {results.box.map50:.4f}')
print(f'mAP50-95:  {results.box.map:.4f}')
print(f'Precision: {results.box.mp:.4f}')
print(f'Recall:    {results.box.mr:.4f}')

Ultralytics 8.4.48  Python-3.13.5 torch-2.8.0+cu126 CPU (AMD Ryzen 9 7945HX with Radeon Graphics)
YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 2.14.1 ms, read: 106.3210.5 MB/s, size: 100.7 KB)
val: Scanning E:\dev\yjwlYOLO\dev\dataset\val\labels.cache... 14861 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 14861/14861 5.7Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 929/929 2.4it/s 6:26<0.5s
                   all      14861     104492      0.691      0.526      0.568      0.429
                   car      14047      87651      0.848      0.798      0.808      0.603
                   bus       2419       2419      0.809      0.553      0.713      0.608
                   van       7296      11750      0.766      0.633      0.657      0.516
                others        666        667      0.613      0.327      0.355      0.285
            m